In [ ]:
import mercury as mr
import flask
name = mr.Text(label="What is your name?")
print(f"Hello {name.value}!")
print("cdecv")

mercury.Text

In [ ]:
import requests
import json


# url = "https://www.dati.lombardia.it/resource/g2hp-ar79.json"
# response = requests.get(url)
# data1 = response.json() 

# with open("sensor_observations.json", "w") as f:
#     json.dump(data1, f, indent=2)

import pandas as pd

csv_file_path = "Dati_sensori_aria_dal_2018_20250525.csv" 
df = pd.read_csv(csv_file_path)

# Convert DataFrame to a list of dictionaries
data1 = df.to_dict(orient="records")

url = "https://www.dati.lombardia.it/resource/ib47-atvt.json"
response = requests.get(url)
data2 = response.json()

with open("sensor_stations.json", "w") as f:
    json.dump(data2, f, indent=2)

In [ ]:
import psycopg2
mydb = psycopg2.connect(
 host="localhost",
 database="SEfG_DB",
 user="postgres",
 password="YourPassword"  # Replace with your password
)
c = mydb.cursor()

In [26]:
c.execute('''CREATE TABLE IF NOT EXISTS sensor_observations
 (idsensore INTEGER,
 data TIMESTAMP,
 valore FLOAT,
 stato TEXT,
 idoperatore INTEGER)''')

c.execute('''
CREATE TABLE IF NOT EXISTS sensor_stations (
    idsensore INTEGER,
    nometiposensore TEXT,
    unitamisura TEXT,
    idstazione INTEGER,
    nomestazione TEXT,
    quota INTEGER,
    provincia TEXT,
    comune TEXT,
    storico TEXT,
    datastart TIMESTAMP,
    utm_nord INTEGER,
    utm_est INTEGER,
    lat DOUBLE PRECISION,
    lng DOUBLE PRECISION,
    location GEOMETRY(POINT, 4326)
)
''')

In [ ]:
# for record in data1:
#     c.execute("""
#         INSERT INTO sensor_observations (idsensore, data, valore, stato, idoperatore)
#         VALUES (%s, %s, %s, %s, %s)
#     """, (
#         record.get("idsensore"),
#         record.get("data"),
#         record.get("valore"),
#         record.get("stato"),
#         record.get("idoperatore")
#     ))

batch_size = len(data1) // 20

for i in range(20):
    start = i * batch_size
    end = (i + 1) * batch_size if i < 19 else len(data1)  # Last batch may be smaller

    batch = data1[start:end]

    values = [
        (
            record.get("idsensore"),
            record.get("data"),
            record.get("valore"),
            record.get("stato"),
            record.get("idoperatore")
        )
        for record in batch
    ]

    c.executemany("""
        INSERT INTO sensor_observations (idsensore, data, valore, stato, idoperatore)
        VALUES (%s, %s, %s, %s, %s)
    """, values)

    mydb.commit()
    print(f"Batch {i + 1}/20 is uploaded: {len(batch)} records")


for record in data2:
    c.execute("""
        INSERT INTO sensor_stations (
            idsensore,
            nometiposensore,
            unitamisura,
            idstazione,
            nomestazione,
            quota,
            provincia,
            comune,
            storico,
            datastart,
            utm_nord,
            utm_est,
            lat,
            lng,
            location
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
                ST_SetSRID(ST_MakePoint(%s, %s), 4326))
    """, (
        record.get("idsensore"),
        record.get("nometiposensore"),
        record.get("unitamisura"),
        record.get("idstazione"),
        record.get("nomestazione"),
        record.get("quota"),
        record.get("provincia"),
        record.get("comune"),
        record.get("storico"),
        record.get("datastart"),
        record.get("utm_nord"),
        record.get("utm_est"),
        record.get("lat"),
        record.get("lng"),
        record["location"]["coordinates"][0], # Longitude
        record["location"]["coordinates"][1]  # Latitude
    ))

Батч 1/20 загружен: 1010082 записей
Батч 2/20 загружен: 1010082 записей
Батч 3/20 загружен: 1010082 записей
Батч 4/20 загружен: 1010082 записей
Батч 5/20 загружен: 1010082 записей
Батч 6/20 загружен: 1010082 записей
Батч 7/20 загружен: 1010082 записей
Батч 8/20 загружен: 1010082 записей
Батч 9/20 загружен: 1010082 записей
Батч 10/20 загружен: 1010082 записей
Батч 11/20 загружен: 1010082 записей
Батч 12/20 загружен: 1010082 записей
Батч 13/20 загружен: 1010082 записей
Батч 14/20 загружен: 1010082 записей
Батч 15/20 загружен: 1010082 записей
Батч 16/20 загружен: 1010082 записей
Батч 17/20 загружен: 1010082 записей
Батч 18/20 загружен: 1010082 записей
Батч 19/20 загружен: 1010082 записей
Батч 20/20 загружен: 1010091 записей


In [28]:
c.execute("""
    ALTER TABLE sensor_stations
    ADD CONSTRAINT sensor_stations_pkey PRIMARY KEY (idsensore);
""")
c.execute("""
    ALTER TABLE sensor_observations
    ADD CONSTRAINT foreignkey_idsensore
    FOREIGN KEY (idsensore)
    REFERENCES sensor_stations(idsensore)
""")

In [29]:
mydb.commit()
c.close()
mydb.close()